In [1]:
import sys, os, inspect
sys.path.insert(0, os.path.abspath(".."))

import torch
from models.st_model import STMultiScaleDiscriminator
from models.roi_discriminator import MultiScaleROITemporalDiscriminator

torch.manual_seed(0)

## 1. Image discriminator -- STMultiScaleDiscriminator (D_A / D_B)

In [2]:
print(inspect.getsource(STMultiScaleDiscriminator))

class STMultiScaleDiscriminator(nn.Module):
    """
    Spatiotemporal multi-scale PatchGAN discriminator.

    Runs num_scales independent factorized CNNs at progressively coarser
    resolutions (AvgPool3d between scales). Returns a list of score maps
    (finest first) — compatible with the existing multi-scale LSGAN losses.

    Input : (B, T, D, H, W)
    Output: list of (B, 1, d, h, w) score tensors, finest first
    """

    def __init__(self, base_ch: int = 64, num_scales: int = 2, temporal_k: int = 3):
        super().__init__()
        self.num_scales = num_scales
        self.downsample = nn.AvgPool3d(3, stride=2, padding=1, count_include_pad=False)
        self.cnns = nn.ModuleList([
            _STScaleCNN(base_ch, temporal_k) for _ in range(num_scales)
        ])

    def forward(self, x: Tensor) -> List[Tensor]:
        """x: (B, T, D, H, W)  →  list of (B, 1, d, h, w)"""
        B, T, D, H, W = x.shape
        x_in = x.unsqueeze(2)       # (B, T, 1, D, H, W)
        out

In [3]:
# Same config as st_v3_ddp_with_roi_time_series_cycleGANS (num_disc_scales=2)
D_B = STMultiScaleDiscriminator(base_ch=64, num_scales=2, temporal_k=3)
n_params = sum(p.numel() for p in D_B.parameters())
print(f"params: {n_params:,}")
print(D_B)

params: 22,059,650
STMultiScaleDiscriminator(
  (downsample): AvgPool3d(kernel_size=3, stride=2, padding=1)
  (cnns): ModuleList(
    (0-1): 2 x _STScaleCNN(
      (blocks): ModuleList(
        (0): FactorizedDiscBlock(
          (spatial): Sequential(
            (0): Conv3d(1, 64, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1))
            (1): LeakyReLU(negative_slope=0.2, inplace=True)
          )
        )
        (1): FactorizedDiscBlock(
          (spatial): Sequential(
            (0): Conv3d(64, 128, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
            (1): InstanceNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
            (2): LeakyReLU(negative_slope=0.2, inplace=True)
          )
        )
        (2): FactorizedDiscBlock(
          (spatial): Sequential(
            (0): Conv3d(128, 256, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
            (1): InstanceNorm3d(256, eps=1e

In [4]:
B, T, D, H, W = 2, 5, 64, 72, 56   # matches PADDED_SPATIAL + in_timepoints
x_hat_b = torch.randn(B, T, D, H, W, requires_grad=True)

scores = D_B(x_hat_b)
print("type:", type(scores), " num scales:", len(scores))
for i, s in enumerate(scores):
    print(f"scale {i}: shape {tuple(s.shape)}  patches/sample={s.shape[-3]*s.shape[-2]*s.shape[-1]}")

loss = sum(s.mean() for s in scores)
loss.backward()
print("backward OK, grad reached x_hat_b:", x_hat_b.grad is not None)

type: <class 'list'>  num scales: 2
scale 0: shape (2, 1, 4, 4, 3)  patches/sample=48
scale 1: shape (2, 1, 2, 2, 1)  patches/sample=4


backward OK, grad reached x_hat_b: True


In [5]:
# Where the two scales diverge: the coarser scale is AvgPool3d'd before its own CNN runs
print(inspect.getsource(D_B.forward))
print(D_B.downsample)  # nn.AvgPool3d applied between scales

    def forward(self, x: Tensor) -> List[Tensor]:
        """x: (B, T, D, H, W)  →  list of (B, 1, d, h, w)"""
        B, T, D, H, W = x.shape
        x_in = x.unsqueeze(2)       # (B, T, 1, D, H, W)
        outputs = []
        for i, cnn in enumerate(self.cnns):
            outputs.append(cnn(x_in))
            if i < self.num_scales - 1:
                # Spatial downsample, keeping T intact
                x_sp = x_in.reshape(B * T, 1, D, H, W)
                x_sp = self.downsample(x_sp)
                _, _, D, H, W = x_sp.shape
                x_in = x_sp.reshape(B, T, 1, D, H, W)
        return outputs

AvgPool3d(kernel_size=3, stride=2, padding=1)


## 2. ROI discriminator -- MultiScaleROITemporalDiscriminator (roi_disc)

In [6]:
print(inspect.getsource(MultiScaleROITemporalDiscriminator))

class MultiScaleROITemporalDiscriminator(nn.Module):
    def __init__(
        self,
        n_rois=400,
        temporal_features=32
    ):
        super().__init__()

        self.n_rois = n_rois
        self.temporal_features = temporal_features

        self.temporal_encoder = nn.Sequential(
            nn.Conv1d(
                1,
                16,
                kernel_size=3,
                padding=1
            ),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv1d(
                16,
                temporal_features,
                kernel_size=3,
                padding=1
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # Separate classifier for every ROI
        self.roi_weight = nn.Parameter(
            torch.randn(n_rois, temporal_features) * 0.02
        )
        self.roi_bias = nn.Parameter(
            torch.zeros(n_rois)
        )

        # Whole-brain classifier
        self.global_head = nn.Sequential(
          

In [7]:
roi_disc = MultiScaleROITemporalDiscriminator(n_rois=400, temporal_features=32)
n_params = sum(p.numel() for p in roi_disc.parameters())
print(f"params: {n_params:,}")
print(roi_disc)

params: 3,292,145
MultiScaleROITemporalDiscriminator(
  (temporal_encoder): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (global_head): Sequential(
    (0): Linear(in_features=12800, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [8]:
n_rois, T_roi = 400, 5
fake_roi = torch.randn(B, n_rois, T_roi, requires_grad=True)

out = roi_disc(fake_roi)
print("keys:", list(out.keys()))
print("global:", tuple(out["global"].shape))   # (B, 1) -- one whole-brain score per sample
print("roi:   ", tuple(out["roi"].shape))       # (B, n_rois) -- one score per ROI per sample

loss = out["global"].mean() + out["roi"].mean()
loss.backward()
print("backward OK, grad reached fake_roi:", fake_roi.grad is not None)

keys: ['roi', 'global']
global: (2, 1)
roi:    (2, 400)
backward OK, grad reached fake_roi: True


In [9]:
# roi_weight/roi_bias are NOT a shared linear layer over all ROIs -- each ROI gets its
# own weight vector, applied via einsum. No mixing across ROIs anywhere in this
# discriminator -- every ROI's temporal encoding is independent (shared temporal_encoder
# weights, but no cross-ROI interaction).
print("temporal_encoder:", roi_disc.temporal_encoder)
print("roi_weight shape:", tuple(roi_disc.roi_weight.shape))   # (n_rois, temporal_features)
print("roi_bias shape:  ", tuple(roi_disc.roi_bias.shape))     # (n_rois,)
print("global_head:", roi_disc.global_head)

temporal_encoder: Sequential(
  (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (1): LeakyReLU(negative_slope=0.2, inplace=True)
  (2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (3): LeakyReLU(negative_slope=0.2, inplace=True)
)
roi_weight shape: (400, 32)
roi_bias shape:   (400,)
global_head: Sequential(
  (0): Linear(in_features=12800, out_features=256, bias=True)
  (1): LeakyReLU(negative_slope=0.2, inplace=True)
  (2): Linear(in_features=256, out_features=1, bias=True)
)
